In [ ]:
from bs4 import BeautifulSoup
import requests
import json
import pandas as pd
import os
from tqdm import tqdm

In [ ]:
def parse_first_stage_cards(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    all_links = []
    
    link_tags = soup.select('a[data-test="cardLinkCover"]')

    for tag in link_tags:
        relative_link = tag.get("href")
        full_link = "https://www.truecar.com" + relative_link if relative_link else None
        
        if full_link:
            car = {"url": full_link}
            sr_span = tag.find("span", class_="sr-only")
            if sr_span:
                raw_title = sr_span.text.strip()
                if raw_title.startswith("View details for "):
                    raw_title = raw_title.replace("View details for ", "", 1).strip()
                car["title"] = raw_title
            all_links.append(car)
    return all_links


In [ ]:
def extract_vehicle_details(html_content, url):
    soup = BeautifulSoup(html_content, "html.parser")
    car_data = {"url": url}

    overview_panel = soup.find(id="tabs_panel__r_5c__0")
    if overview_panel:
        details_container = overview_panel.find("div", {"data-test": "vehicleDetailsOverviewDetails"})
        if details_container:
            items = details_container.find_all("div", class_="flex items-center justify-between")
            for item in items:
                spans = item.find_all("span")
                if len(spans) >= 2:
                    key = spans[0].text.strip().lower().replace(" ", "_")
                    val = spans[1].text.strip()
                    if key:
                        car_data[key] = val
        highlights_container = overview_panel.find("div", {"data-test": "vehicleDetailsOverviewKeyHighlights"})
        if highlights_container:
            highlights = highlights_container.find_all("div", {"data-test": "vehicleDetailsOverviewKeyHighlight"})
            car_data["overview_highlights"] = ", ".join([h.text.strip() for h in highlights])

    features_panel = soup.find(id="tabs_panel__r_5c__1")
    if features_panel:
        features_container = features_panel.find("div", {"data-test": "vehicleDetailsKeyFeatures"})
        if features_container:
            categories = features_container.find_all("div", class_="flex gap-x-8 items-start")
            for cat in categories:
                title_el = cat.find("div", class_="text-18")
                if title_el:
                    title = title_el.text.strip().lower().replace(" ", "_").replace("&", "and")
                    lis = cat.find_all("li")
                    car_data[f"feature_{title}"] = ", ".join([li.text.strip() for li in lis])

    specs_panel = soup.find(id="tabs_panel__r_5c__2")
    if specs_panel:
        categories = specs_panel.find_all("div", class_="text-12")
        for cat in categories:
            title_el = cat.find("div", class_="text-18")
            if title_el:
                lis = cat.find_all("li")
                for li in lis:
                    span = li.find("span")
                    div = li.find("div", class_="flex text-12")
                    if span and div:
                        car_data[f"spec_{span.text.strip().lower().replace(' ', '_')}"] = div.text.strip()

    history_panel = soup.find(id="tabs_panel__r_5c__3")
    if history_panel:
        history_tab = history_panel.find(id="history-tab")
        if history_tab:
            lis = history_tab.find_all("li")
            history_items = []
            for li in lis:
                text = li.text.strip().replace(" \n", "").strip()
                history_items.append(text)
            car_data["history"] = ", ".join(history_items)

    return car_data


In [ ]:
def scrape_city_state(city, state, max_pages=5):
    city_slug = city.lower().replace(" ", "-")
    state_slug = state.lower()
    location_key = f"{city_slug}_{state_slug}"
    os.makedirs("./data", exist_ok=True)

    all_links = []
    for i in tqdm(range(1, max_pages + 1), desc=f"{location_key} - Search Pages"):
        url = f"https://www.truecar.com/used-cars-for-sale/listings/location-{city_slug}-{state_slug}/?page={i}"
        response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
        # Note: If this returns 403, replace `requests.get` with Selenium or cloudscraper
        page_links = parse_first_stage_cards(response.text)
        all_links.extend(page_links)

    json_path = f"./data/truecar_links_{location_key}.json"
    with open(json_path, "w") as f:
        json.dump(all_links, f, indent=2)

    print(f"Saved {len(all_links)} links for {city.title()}, {state.upper()}")
    
    results = []
    for entry in tqdm(all_links, desc="Scraping car details"):
        response = requests.get(entry["url"], headers={'User-Agent': 'Mozilla/5.0'})
        car_info = extract_vehicle_details(response.text, entry["url"])
        car_info["title"] = entry.get("title", "")
        results.append(car_info)

    df = pd.DataFrame(results)
    output_csv = f"./data/truecar_details_{location_key}.csv"
    output_json = f"./data/truecar_details_{location_key}.json"
    df.to_csv(output_csv, index=False)
    with open(output_json, "w") as f:
        json.dump(results, f, indent=2)
    print(f"Saved complete details to {output_csv}")


In [ ]:
if __name__ == "__main__":
    # Cities within a ~100 mile radius of Boston
    cities = [
        ("Boston", "MA"),
        ("Worcester", "MA"),
        ("Providence", "RI"),
        ("Manchester", "NH"),
        ("Nashua", "NH"),
        ("Lowell", "MA"),
        ("Hartford", "CT"),
        ("Springfield", "MA")
    ]
    for city, state in cities:
        scrape_city_state(city, state, max_pages=5)
